# Clean speech (single speaker)

Part of the per-condition split of `cortical_analysis.ipynb`; runs on its own.
Shared settings (models, fit parameters, ROI, peak windows, plot window) are in
`parameters.py`.

**Fits and model tests are read from saved results**
(`dataset/cocktail/bids/derivatives/results`), not from eelbrain's cache. They are
computed on the cluster (`sbatch --account=def-ibruce cluster/run_analysis.sbatch`)
and fetched to the Mac with `bash cluster/fetch_results.sh`. A missing result stops
with a message saying which one; set `COCKTAIL_RESULTS=compute` before starting
Jupyter to fit it locally instead.

**Peaks:** each subject's peak is searched within +-25 ms of where that condition's
group-average waveform peaks inside the search window (`add_anchored_peak`).

**Per-bin TRF plots** (`plot_trf_bins`): one line per frequency bin in its own colour
with a band of +-1 SD across subjects (`BIN_ERROR` in `parameters.py`), the mean of
the bins as the thick black line, and the bins' centre frequencies on the right.

In [ ]:
from parameters import *

## Clean speech (single speaker)

In [ ]:
# Loaded once here, reused by every STRF/peak/waveform cell below in
# this section - each one just reads a different predictor column out
# of this same fit instead of re-running the (expensive) TRF fit again.
data_full = load_trfs_saved(FULL_CLEAN, 'clean')

In [ ]:
# ROI sensors
p = plot.SensorMap(data_full['gammatone_8'].sensor, mark=ROI)

<small>

**Brodbeck et al. 2020: Clean, onset unique contribution**

- Onsets add their own predictive power: t_max = 12.00, p ≤ 0.001
- Found in auditory cortex on both sides (Heschl's gyrus and superior temporal gyrus)
- Almost every subject shows a gain
- Best overall model fit: r = 0.143

</small>

In [ ]:
# Onset unique contribution (clean)
# Brodbeck et al. 2020 ran this comparison too: single-speaker onset unique contribution, t = 12.00, p <= .001.
data, diff, result = compute_unique_contribution(FULL_CLEAN, onset(ATTEND), 'clean')
show_topomap_trio(data, diff, result, 'Onset', 'clean', onset(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Onset (clean): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Onset', 'clean')

<small>

**Brodbeck et al. 2020: Clean, envelope unique contribution**

- Envelope adds its own predictive power: t_max = 9.39, p ≤ 0.001
- Same place: auditory cortex on both sides
- Almost every subject shows a gain

</small>

In [ ]:
# Envelope unique contribution (clean)
# Brodbeck et al. 2020 ran this comparison too: single-speaker envelope unique contribution, t = 9.39, p <= .001.
data, diff, result = compute_unique_contribution(FULL_CLEAN, envelope(ATTEND), 'clean')
show_topomap_trio(data, diff, result, 'Envelope', 'clean', envelope(ATTEND))

In [ ]:
display(result)
ttest = test.TTestOneSample(f'roi_{METRIC}', data=diff, tail=1)
print(f"Envelope (clean): ROI unique contribution t({ttest.df}) = {ttest.t:.3f}, p = {ttest.p:.4f}")

In [ ]:
p = show_subject_swarm(diff, 'Envelope', 'clean')

<small>

**Brodbeck et al. 2020: Clean, onset STRF**

- Clear two-peak shape: a positive peak, then a negative peak
- Same shape in every frequency band
- Bigger at lower frequencies

</small>

In [ ]:
# Onset STRF (clean)
strf = extract_strf(data_full, 'gammatone_on_8')
p = plot.Array(strf * 1e3, title='Onset STRF (clean)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

<small>

**Brodbeck et al. 2020: Clean, envelope STRF**

- Weaker and less clear-cut than the onset STRF (onsets explain much of what usually looks like an envelope response)
- Fitted without onsets, the envelope STRF shows the usual clear peaks, with bigger amplitude
- Bigger at lower frequencies
- Shown from -50 to 450 ms "because of occasional edge artifacts" (fit window -100 to 500 ms)

</small>

In [ ]:
# Envelope STRF (clean)
strf = extract_strf(data_full, 'gammatone_8')
p = plot.Array(strf * 1e3, title='Envelope STRF (clean)', interpolation=STRF_INTERPOLATION, xlim=PLOT_WINDOW)

<small>

**Brodbeck et al. 2020: Clean, onset peaks**

- Positive peak: 65 ms (average over subjects)
- Negative peak: 126 ms
- Peak search window: 20 to 200 ms (this notebook: 40 to 200 ms)

</small>

In [ ]:
# Onset peak amplitude & timing (clean)
# Brodbeck et al. 2020 ran this comparison too: onset STRF positive peak ~65ms, negative peak ~126ms.
data = extract_trf(data_full, 'gammatone_on_8')
data = add_anchored_peak(data, 'positive', ONSET_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', ONSET_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Onset (clean) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Onset (clean) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# The TRF the peaks above were measured from, per frequency bin
p = plot_trf_bins(data_full, 'gammatone_on_8', 'Onset TRF (clean)')

<small>

**Brodbeck et al. 2020: Clean, envelope peaks**

- No envelope peak latencies reported (the envelope STRF has no clear peaks)

</small>

In [ ]:
# Envelope peak amplitude & timing (clean)
data = extract_trf(data_full, 'gammatone_8')
data = add_anchored_peak(data, 'positive', ENVELOPE_POSITIVE_PEAK_WINDOW, kind='max')
data = add_anchored_peak(data, 'negative', ENVELOPE_NEGATIVE_PEAK_WINDOW, kind='min')
print(f"Envelope (clean) positive peak: {data['positive_time'].mean() * 1e3:.0f} ms, amp {data['positive_amp'].mean():.4g}")
print(f"Envelope (clean) negative peak: {data['negative_time'].mean() * 1e3:.0f} ms, amp {data['negative_amp'].mean():.4g}")

In [ ]:
# The TRF the peaks above were measured from, per frequency bin
p = plot_trf_bins(data_full, 'gammatone_8', 'Envelope TRF (clean)')